# Step 1: Data Preparation

Download ChEMBL, filter Ki/EC50, clean SMILES, scaffold-split, augment, tokenise, and save as HDF5.

In [1]:
from molrl.cheminformatics import fetch_chembl_bioactivity, cleaning_pipeline
from molrl.tokenizer import is_tokenizable
import pandas as pd
import numpy as np
import os
from rdkit import Chem
from rdkit.Chem import Descriptors  

In [2]:
# Serotonin 5-HT2A receptor (CHEMBL224) is a cool target for antidepressants/antiphsychotics and has a large number of bioactivity records in ChEMBL, making it a good candidate for this project.
raw_chembl224_ki_data = fetch_chembl_bioactivity(target_chembl_id='CHEMBL224', standard_types=['Ki'], relations=['='])

In [3]:
# Remove datapoints with non-NaN data_validity_comment values
subset_chembl224_ki_data = raw_chembl224_ki_data[raw_chembl224_ki_data['data_validity_comment'].isna()]

# convert Ki values to floats
subset_chembl224_ki_data['value'] = pd.to_numeric(subset_chembl224_ki_data['value'], errors='coerce')

# Take the average of multiple measurements for the same compound (molecule_chembl_id), create a mean and std column for the Ki values
subset_chembl224_ki_data = subset_chembl224_ki_data.groupby('molecule_chembl_id').agg(
    smiles=('smiles', 'first'),
    mean_Ki=('value', 'mean'),
    std_Ki=('value', 'std'),
    count_Ki=('value', 'count')
).reset_index()

# Keep single measurements (no std by definition) and multi-measurements with CV < 0.5
subset_chembl224_ki_data = subset_chembl224_ki_data[
    subset_chembl224_ki_data['count_Ki'].eq(1) |
    (subset_chembl224_ki_data['std_Ki'] / subset_chembl224_ki_data['mean_Ki'] < 0.5)
]

# keep the smiles and mean_Ki columns, rename mean_Ki to 'Ki [nM]'
subset_chembl224_ki_data = subset_chembl224_ki_data[['smiles', 'mean_Ki']].rename(columns={'mean_Ki': 'Ki [nM]'})

# add a pki column which is the negative log of the Ki value in molar units (i.e. -log10(Ki [nM] * 1e-9))
subset_chembl224_ki_data['pKi'] = -np.log10(subset_chembl224_ki_data['Ki [nM]'] * 1e-9)

print(subset_chembl224_ki_data)


                                                 smiles   Ki [nM]       pKi
2        CN1CCC(c2c[nH]c3ccc(NC(=O)c4ccc(F)cc4)nc23)CC1  4800.000  5.318759
3     S=C(NCCCc1c[nH]cn1)NCCCN(Cc1ccc(Cl)c(Cl)c1)c1c...   202.300  6.694004
4                Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1  3183.000  5.497163
5                              NCCc1ccc(CCCc2ccccc2)cc1    60.000  7.221849
6      Nc1nc(SCCN2CCN(Cc3ccccc3Cl)CC2)nc(N)c1Cc1ccccc1O    22.800  7.642065
...                                                 ...       ...       ...
4684  O=S(=O)(Nc1ccc2nccc(N3CCNCC3)c2c1)c1ccc(-c2ccc...   501.190  6.299998
4685          Cc1ccc(OC2CCN(CCN3CCCc4ccccc4C3=O)CC2)cc1    54.950  7.260032
4686              O=c1cc(N2CCOCC2)oc2c(-c3ccccc3)cccc12   799.000  6.097453
4687      O=C1c2ccccc2CCCN1CCN1CCC(n2ccc3cc(F)ccc32)CC1     6.918  8.160019
4688         O=C1c2ccccc2CCCN1CCN1CCC(n2ccc3ccccc32)CC1    19.500  7.709965

[4561 rows x 3 columns]


In [5]:
# Clean the molecules. An important side note is that we flatten stereochemistry.
standardized_smiles, statuses = cleaning_pipeline(subset_chembl224_ki_data['smiles'].tolist())
subset_chembl224_ki_data['standardized_smiles'] = standardized_smiles
subset_chembl224_ki_data['cleaning_status'] = statuses
subset_chembl224_ki_data = subset_chembl224_ki_data[(subset_chembl224_ki_data['cleaning_status'] == 'pass')]

# keep only tokanizable molecules
subset_chembl224_ki_data['tokenizable'] = subset_chembl224_ki_data['standardized_smiles'].apply(is_tokenizable)
subset_chembl224_ki_data = subset_chembl224_ki_data[(subset_chembl224_ki_data['tokenizable'])]

# remove the original smiles column
cleaned_chembl224_ki_data = subset_chembl224_ki_data.drop(columns=['smiles', 'cleaning_status', 'tokenizable'])

cleaned_chembl224_ki_data.to_csv('chembl224_ki_cleaned.csv', index=False)
print(cleaned_chembl224_ki_data)

Cleaning SMILES:   0%|          | 0/45 [00:00<?, ?it/s]

       Ki [nM]       pKi                                standardized_smiles
2     4800.000  5.318759     CN1CCC(c2c[nH]c3ccc(NC(=O)c4ccc(F)cc4)nc23)CC1
3      202.300  6.694004  S=C(NCCCc1c[nH]cn1)NCCCN(Cc1ccc(Cl)c(Cl)c1)c1c...
4     3183.000  5.497163             Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1
5       60.000  7.221849                           NCCc1ccc(CCCc2ccccc2)cc1
6       22.800  7.642065   Nc1nc(SCCN2CCN(Cc3ccccc3Cl)CC2)nc(N)c1Cc1ccccc1O
...        ...       ...                                                ...
4684   501.190  6.299998  O=S(=O)(Nc1ccc2nccc(N3CCNCC3)c2c1)c1ccc(-c2ccc...
4685    54.950  7.260032          Cc1ccc(OC2CCN(CCN3CCCc4ccccc4C3=O)CC2)cc1
4686   799.000  6.097453              O=c1cc(N2CCOCC2)oc2c(-c3ccccc3)cccc12
4687     6.918  8.160019      O=C1c2ccccc2CCCN1CCN1CCC(n2ccc3cc(F)ccc32)CC1
4688    19.500  7.709965         O=C1c2ccccc2CCCN1CCN1CCC(n2ccc3ccccc32)CC1

[4486 rows x 3 columns]


In [6]:
from molrl.cheminformatics import scaffold_split
from sklearn.model_selection import train_test_split

# Scaffold split 80/20 → (train+val) / test — no scaffold leakage across splits
train_val_data, test_data = scaffold_split(cleaned_chembl224_ki_data, smiles_col='standardized_smiles', test_size=0.2)

# Random split train+val 80/20 → train / val
train_data, val_data = train_test_split(train_val_data, test_size=0.2, random_state=42)
train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}, Total: {len(cleaned_chembl224_ki_data)}")

train_data.to_csv('chembl224_ki_train.csv', index=False)
val_data.to_csv('chembl224_ki_val.csv', index=False)
test_data.to_csv('chembl224_ki_test.csv', index=False)

Train: 2871, Val: 718, Test: 897, Total: 4486


In [7]:
def download_chembl():
    chembl_url = 'https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_36_chemreps.txt.gz'

    if not os.path.exists('chembl_36_chemreps.csv'):
        chembl_data = pd.read_csv(chembl_url, sep='\t', compression='gzip')
        chembl_data.to_csv('chembl_36_chemreps.csv', index=False)

    chembl_data = pd.read_csv('chembl_36_chemreps.csv')

    return chembl_data

chembl_data = download_chembl()

print(chembl_data.head())

      chembl_id                                   canonical_smiles  \
0  CHEMBL153534                       Cc1cc(-c2csc(N=C(N)N)n2)cn1C   
1  CHEMBL440060  CC[C@H](C)[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H...   
2  CHEMBL440245  CCCC[C@@H]1NC(=O)[C@@H](NC(=O)[C@H](CC(C)C)NC(...   
3  CHEMBL440249  CC(C)C[C@@H]1NC(=O)CNC(=O)[C@H](c2ccc(O)cc2)NC...   
4  CHEMBL405398             Brc1cccc(Nc2ncnc3ccncc23)c1NCCN1CCOCC1   

                                      standard_inchi  \
0  InChI=1S/C10H13N5S/c1-6-3-7(4-15(6)2)8-5-16-10...   
1  InChI=1S/C123H212N44O34S/c1-19-63(12)96(164-11...   
2  InChI=1S/C160H268N50O41/c1-23-27-41-95-134(228...   
3  InChI=1S/C124H154ClN21O39/c1-57(2)48-81-112(17...   
4  InChI=1S/C19H21BrN6O/c20-15-2-1-3-17(18(15)22-...   

            standard_inchi_key  
0  MFRNFCWYPYSFQQ-UHFFFAOYSA-N  
1  RSEQNZQKBMRQNM-VRGFNVLHSA-N  
2  FTKBTEIKPOYCEX-OZSLQWTKSA-N  
3  UYSXXKGACMHPIM-KFGDMSGDSA-N  
4  VDSXZXJEWIWBCG-UHFFFAOYSA-N  


In [ ]:
# keep only molecules that pass the cleaning pipeline, we skip normalizing tautomers because that takes waaaay too long
standardized_smiles, statuses = cleaning_pipeline(chembl_data['canonical_smiles'].tolist(), skip_tautomer_canonicalization=True)
chembl_data['standardized_smiles'] = standardized_smiles
chembl_data['cleaning_status'] = statuses
chembl_data = chembl_data[(chembl_data['cleaning_status'] == 'pass')]

# keep all molecules that are tokenizable
chembl_data['tokenizable'] = chembl_data['standardized_smiles'].apply(is_tokenizable)
chembl_data = chembl_data[(chembl_data['tokenizable'])]

# keep molecules with reasonable molecular weights, between 50 and 750 daltons
chembl_data['molecular_weight'] = chembl_data['standardized_smiles'].apply(lambda smi: Descriptors.MolWt(Chem.MolFromSmiles(smi)))
chembl_data = chembl_data[(chembl_data['molecular_weight'] >= 50) & (chembl_data['molecular_weight'] <= 750)]

# keep molecules with reasonable number of heavy atoms, >5
chembl_data['num_heavy_atoms'] = chembl_data['standardized_smiles'].apply(lambda smi: Chem.MolFromSmiles(smi).GetNumHeavyAtoms())
chembl_data = chembl_data[chembl_data['num_heavy_atoms'] > 5]

# only keep the standardized_smiles column.
chembl_data_clean = chembl_data[['standardized_smiles']]
chembl_data_clean.to_csv('chembl_36_clean.csv', index=False)

print(chembl_data_clean)

In [9]:
chembl_data_clean = pd.read_csv('chembl_36_clean.csv')

In [ ]:
# I want to remove all molecules from the pretraining data that are too similar to the test set molecules, 
# so I calculate the Tanimoto similarity of ECFP4 fingerprints between all cleaned ChEMBL molecules and the 
# test set molecules, and remove any ChEMBL molecule that has a Tanimoto similarity > 0.5 to any test set molecule.

from molrl.cheminformatics import calculate_ecfps, tanimoto_one_to_many
import jax.numpy as jnp
from tqdm.asyncio import tqdm
from jax import vmap

chembl224_ki_test_fps = jnp.array(calculate_ecfps(test_data['standardized_smiles'].tolist(), radius=2, nbits=1024)).astype('float16')
chembl_fps = jnp.array(calculate_ecfps(chembl_data_clean['standardized_smiles'].tolist(), radius=2, nbits=1024)).astype('float16')

# compute the maximum Tanimoto similarity of each ChEMBL molecule to any test set molecule
max_tanimoto_to_set = vmap(tanimoto_one_to_many, (0, None))
chembl_data_clean['max_similarity_to_chembl224_ki'] = max_tanimoto_to_set(chembl_fps, chembl224_ki_test_fps).max(axis=1)

# remove any ChEMBL molecule that has a Tanimoto similarity > 0.4 to any test set molecule
chembl_data_clean = chembl_data_clean[chembl_data_clean['max_similarity_to_chembl224_ki'] <= 0.4]

# Only keep the standardized_smiles column
chembl_data_clean = chembl_data_clean[['standardized_smiles']]
chembl_data_clean.to_csv('chembl_36_clean_filtered.csv', index=False)

In [22]:
# randomly split the cleaned ChEMBL data into 90/5/5 train/val/test splits
train_val_data, chembl_test_data = train_test_split(chembl_data_clean, test_size=0.05, random_state=42)
chembl_train_data, chembl_val_data = train_test_split(chembl_data_clean, test_size=0.0526, random_state=42)  # 0.0526 * 0.95 ≈ 0.05

chembl_train_data.to_csv('chembl_train.csv', index=False)
chembl_val_data.to_csv('chembl_val.csv', index=False)
chembl_test_data.to_csv('chembl_test.csv', index=False)

In [ ]:
# perform SMILES agumentation on all pretraining data and on the training and validation splits, but not the test split (to avoid data leakage)